# Playground for Open WebUI API

https://docs.openwebui.com/reference/api-endpoints/

In [ ]:
import os
import requests
from dotenv import load_dotenv
import time

load_dotenv()  # Automatically finds .env file
openwebui_api_key = os.getenv('OPENWEBUI_API_KEY')


BASE_URL = "http://localhost:3000/api/v1"
API_KEY = openwebui_api_key
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

In [ ]:
def upload_file(file_path):
    url = 'http://localhost:3000/api/v1/files/'
    headers = {**HEADERS, 'Accept': 'application/json'}
    files = {'file': open(file_path, 'rb')}
    response = requests.post(url, headers=headers, files=files)
    return response.json()


def wait_for_file_processing(token, file_id, timeout=300, poll_interval=2):
    """
    Wait for a file to finish processing.
    
    Returns:
        dict: Final status with 'status' key ('completed' or 'failed')
    
    Raises:
        TimeoutError: If processing doesn't complete within timeout
    """
    url = f'http://localhost:3000/api/v1/files/{file_id}/process/status'
    headers = {'Authorization': f'Bearer {token}'}
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        response = requests.get(url, headers=headers)
        result = response.json()
        status = result.get('status')
        
        if status == 'completed':
            return result
        elif status == 'failed':
            raise Exception(f"File processing failed: {result.get('error')}")
        
        time.sleep(poll_interval)
    
    raise TimeoutError(f"File processing did not complete within {timeout} seconds")


def create_knowledge_base(name, description):
    json_headers = {**HEADERS, "Content-Type": "application/json"}
    payload = {
        "name": name,
        "description": description,
    }
    res = requests.post(f"{BASE_URL}/knowledge/create", headers=json_headers, json=payload)
    if res.status_code == 200:
        print(f"\nKnowledge Base '{name}' created! ID: {res.json()['id']}")
    else:
        print(f"\nFailed to create Knowledge Base: {res.text}")
    return res.json()

def add_file_to_knowledge(knowledge_id, file_id):
    url = f'http://localhost:3000/api/v1/knowledge/{knowledge_id}/file/add'
    json_headers = {**HEADERS, "Content-Type": "application/json"}
    data = {'file_id': file_id}
    response = requests.post(url, headers=json_headers, json=data)
    return response.json()


## Creating knowledge base and adding files

In [ ]:
kb_name = "Dos ciclos"
kb_desc = "Falta una descripción."

kb_json = create_knowledge_base(
    name=kb_name,
    description=kb_desc,
)

# Knowledge Base 'Dos ciclos' created! ID: 5c99f2d6-51ed-4a9b-81fd-4740259a8993

In [ ]:
kb_json

# {'id': '5c99f2d6-51ed-4a9b-81fd-4740259a8993',
#  'user_id': 'c6796723-1e34-41af-8509-293ab52662d1',
#  'name': 'Dos ciclos',
#  'description': 'Falta una descripción.',
#  'meta': None,
#  'access_grants': [],
#  'created_at': 1786049434,
#  'updated_at': 1786049434,
#  'files': None}


In [ ]:
file_name = "TD-actividad-01-ciclos.pdf"
folder_name = "./other"
# folder_path = os.path.expanduser(f"~/Downloads/{folder_name}/")
file_path = os.path.join(folder_name, file_name)
print(f"Uploading file: {file_path}")

In [ ]:
file_json = upload_file(file_path)
print(file_json)

# {'id': '14ceced4-e59d-4a3b-ab31-6e5f66b82386', 'user_id': 'c6796723-1e34-41af-8509-293ab52662d1', 'hash': None, 'filename': 'TD-actividad-01-ciclos.pdf', 'data': {'status': 'pending'}, 'meta': {'name': 'TD-actividad-01-ciclos.pdf', 'content_type': None, 'size': 60574, 'file_hash': '5ebef3abb39c8c45dd62f08d6074fd20421fd12183702005ba9c090e0280b2b4', 'data': {}}, 'created_at': 1786049961, 'updated_at': 1786049961, 'status': True, 'path': '/app/backend/data/uploads/14ceced4-e59d-4a3b-ab31-6e5f66b82386_TD-actividad-01-ciclos.pdf'}

In [ ]:
# Wait for the file to finish processing
url = f'http://localhost:3000/api/v1/files/{file_json['id']}/process/status'

response = requests.get(url, headers={**HEADERS})
result = response.json()
status = result.get('status')
print(status)

# completed

In [ ]:
added_json = add_file_to_knowledge(kb_json['id'], file_json['id'])
print(added_json)

# {'id': '5c99f2d6-51ed-4a9b-81fd-4740259a8993', 'user_id': 'c6796723-1e34-41af-8509-293ab52662d1', 'name': 'Dos ciclos', 'description': 'Falta una descripción.', 'meta': None, 'access_grants': [], 'created_at': 1786049434, 'updated_at': 1786049434, 'files': [{'id': '14ceced4-e59d-4a3b-ab31-6e5f66b82386', 'hash': '4afee8b5eb083d0f46c6cc6f3e99ad80a7373f083b6c0778bb5f5bc9cd60a7a7', 'meta': {'name': 'TD-actividad-01-ciclos.pdf', 'content_type': None, 'size': 60574, 'file_hash': '5ebef3abb39c8c45dd62f08d6074fd20421fd12183702005ba9c090e0280b2b4', 'data': {}, 'collection_name': '5c99f2d6-51ed-4a9b-81fd-4740259a8993'}, 'created_at': 1786049961, 'updated_at': 1786050252}], 'write_access': False}

## Create and update a model

In [ ]:
def create_model(name):
    json_headers = {**HEADERS, "Content-Type": "application/json"}
    payload = {
        "name": name,
        "description": description,
    }
    res = requests.post(f"{BASE_URL}/models/create", headers=json_headers, json=payload)
    if res.status_code == 200:
        print(f"\nModel '{name}' created! ID: {res.json()['id']}")
    else:
        print(f"\nFailed to create Model: {res.text}")
    return res.json()

payload = {
    "id": "custom-assistant-id",
    "name": "My Custom Assistant",
    "meta": {
      "suggestion_prompts": [
        {"id": "s1", "content": "Help me draft an email."}
      ],
      "profile_image_url": "https://example.com"
    },
    "params": {
      "system": "You are a specialized AI assistant that speaks strictly in a professional corporate tone."
    },
    "base_model_id": "llama3:latest"
  }


In [ ]:
payload = {
    "id": "custom-assistant-id",
    "name": "My Custom Assistant",
    # "meta": {
    #   "suggestion_prompts": [
    #     {"id": "s1", "content": "Help me draft an email."}
    #   ],
    # },
    "params": {
      "system": "You are a specialized AI assistant that speaks strictly in a professional corporate tone."
    },
    "base_model_id": "llama3:latest"
  }